# Agentic RAG: Self-Correction Loop using LangGraph

This implementation features a state-machine based RAG system. Unlike traditional linear pipelines, this architecture uses a feedback loop to validate document relevance and generation accuracy, minimizing hallucinations and improving reliability in production environments.

In [ ]:
import os
import logging
from typing import List, TypedDict, Annotated
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AgenticRAG")

### State Definition
The `GraphState` manages the lifecycle of the query, including retrieval attempts to prevent infinite loops.

In [ ]:
class GraphState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    relevance_score: int
    retrieval_attempts: int

### Core Logic: Nodes
Each node represents a discrete step in the RAG process.

In [ ]:
def retrieve(state: GraphState):
    logger.info("Action: Retrieving contextually relevant documents")
    # Note: 'retriever' object must be initialized with your vector store
    docs = retriever.invoke(state["question"])
    return {"documents": docs, "retrieval_attempts": state.get("retrieval_attempts", 0) + 1}

def grade_relevance(state: GraphState):
    logger.info("Action: Grading document relevance")
    # Implementation for LLM-as-a-judge goes here
    return {"relevance_score": 8} 

def generate(state: GraphState):
    logger.info("Action: Generating final response")
    return {"answer": "Synthesized response based on validated context."}

### Graph Orchestration
The workflow connects nodes through conditional edges to enable self-correction.

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_relevance", grade_relevance)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_relevance")

def decide_next_step(state: GraphState):
    if state["relevance_score"] < 6 and state["retrieval_attempts"] < 3:
        return "retrieve"
    return "generate"

workflow.add_conditional_edges(
    "grade_relevance",
    decide_next_step,
    {"retrieve": "retrieve", "generate": "generate"}
)

workflow.add_edge("generate", END)
app = workflow.compile()